In [1]:
import torch
import torch.nn as nn
import random
import os

In [2]:
EPSILON = 1 
EPSILON_DECAY = 0.095
ACTION_SPACE = [0,1,2,3]
GAMMA = 0.99
LR = 1e-3
INPUT_SIZE = 8
OUTPUT_SIZE = 5 
HIDDEN_SIZE = 64 
EPOCHS = 1000
FREQ_OF_TARGET_NN_UPDATE = 20
REPLAY_MEMORY_SIZE = 500000

In [3]:
def policy(epsilon, action_space, highest_estimated_action):
    if random.random() < epsilon:
        return random.randint(0, len(action_space) - 1)
    else: 
        return highest_estimated_action

In [4]:
#[head_x, head_y, direction, food_x, food_y, danger_straight, danger_right, danger_left]
class DQN_scaled_ff(nn.Module):
    def __init__(self, input_size, output_size, hidden_size) -> None:
        super().__init__()
        hidden_size1 = int(hidden_size * 2)
        hidden_size2 = int(hidden_size / 2)
        
        self.input_linear = nn.Linear(input_size, hidden_size)
        self.linear_sequence1 = nn.Sequential(
            nn.Linear(hidden_size, hidden_size1),
            nn.ReLU(), 
            nn.Linear(hidden_size1, hidden_size1),
            nn.Linear(hidden_size1, hidden_size1),
            nn.ReLU(),
            nn.Linear(hidden_size1, hidden_size1),
            nn.Linear(hidden_size1, hidden_size1),
            nn.ReLU(),
            nn.Linear(hidden_size1, hidden_size1)
        )
        self.linear_sequence2 = nn.Sequential(
            nn.Linear(hidden_size1, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, hidden_size2),
            nn.Linear(hidden_size2, hidden_size2),
            nn.ReLU(), 
            nn.Linear(hidden_size2, hidden_size2),
            nn.Linear(hidden_size2, hidden_size2),
            nn.ReLU(), 
            nn.Linear(hidden_size2, hidden_size2)
        )
        self.output_linear = nn.Linear(hidden_size2, output_size)

    def forward(self, x):
        x = self.input_linear(x) 
        x = self.linear_sequence1(x)
        x = self.linear_sequence2(x)
        x = self.output_linear(x)
        return x
    
    def save(self, file_name="model.pth"):
        model_folder_path = "./model"
        if not os.path.exists(model_folder_path):
            os.makedirs(model_folder_path)
        
        file_name = os.path.join(model_folder_path, file_name)
        torch.save(self.state_dict(), file_name)

In [5]:
device = "mps"

In [6]:
score = 0
steps = 0

In [7]:
active_nn = DQN_scaled_ff(INPUT_SIZE, OUTPUT_SIZE, HIDDEN_SIZE).to(device=device)
active_nn.load_state_dict(torch.load("model/scaled_ff_model.pth"))

<All keys matched successfully>

In [ ]:
from game import Snake
evaluation_game = Snake(evaluation=True)
while True:
    initial_state = evaluation_game.get_state()
    action_pred = active_nn(torch.tensor(initial_state, dtype=torch.float32).to(device))
    action = policy(0, ACTION_SPACE, torch.argmax(action_pred))
    done = evaluation_game.move_with_action(action=action)
    print(done) 
    steps += 1
    if evaluation_game.get_immediate_reward() == 1:
        score += 1
    if done == 1:
        break

pygame 2.6.1 (SDL 2.28.4, Python 3.12.9)
Hello from the pygame community. https://www.pygame.org/contribute.html
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
1


: 

In [1]:
from game import Snake
evaluation_game = Snake()

evaluation_game.get_reward()

pygame 2.6.1 (SDL 2.28.4, Python 3.12.9)
Hello from the pygame community. https://www.pygame.org/contribute.html
[(4, 20), (5, 20), (6, 20), (7, 20), (8, 20), (9, 20), (10, 20), (11, 20), (12, 20), (13, 20), (14, 20), (15, 20)]


0

In [3]:
evaluation_game.get_food_position()

(35, 24)

In [36]:
import random
import redis
import pickle
import torch

r = redis.Redis(host="localhost", port=6379, db=0)
batch_size = 5 
initial_state = []
final_state = []
action = []
action_pred = []
reward = []
done = []

replay_memory_size = r.dbsize()
random_keys = random.sample(range(1, replay_memory_size + 1), k=batch_size)
print(f"Random keys: {random_keys}")

for key in random_keys:
    experience = pickle.loads(r.get(key))
    initial_state += [experience[0]]
    action += [experience[1]]
    final_state += [experience[2]]
    reward += [experience[3]]
    done += [experience[4]]

initial_state = torch.tensor(initial_state, dtype=torch.float32, device="mps")
final_state = torch.tensor(final_state, dtype=torch.float32, device="mps")
action_pred = torch.tensor(action_pred, dtype=torch.float32, device="mps")
action = torch.tensor(action, device="mps", dtype=torch.long).unsqueeze(0).T
reward = torch.tensor(reward, dtype=torch.float32, device="mps")
done = torch.tensor(done, dtype=torch.float32, device="mps")

Random keys: [13283286, 9053355, 7291670, 5307007, 6058039]


In [37]:
# q_target shape: [5]
# active_prediction shape: [5]

In [42]:
def loss_function(active_prediction, gamma, immediate_reward, action, done, final_state, target_nn):
    active_prediction = active_prediction.gather(dim=1, index=action).squeeze(1)
    next_state_q_values = target_nn(final_state) 
    next_state_value_target = torch.max(next_state_q_values, dim=1)[0]
    q_target = immediate_reward + (1.0 - done) * (gamma * next_state_value_target) 

    loss = (q_target - active_prediction) ** 2
    return loss.mean()

In [43]:
import torch.nn as nn
import torch.nn.functional as F

EPSILON = 1 
EPSILON_DECAY = 0.095
ACTION_SPACE = [0,1,2,3,4]
GAMMA = 0.99
LR = 1e-3
INPUT_SIZE = 8
OUTPUT_SIZE = 5 
FREQ_OF_TARGET_NN_UPDATE = 20
REPLAY_MEMORY_SIZE = 15500000
HIDDEN_SIZE = 64 
EPOCHS = 1000000
EPISODE_ID = 0

class DQN_base_ff(nn.Module):
    def __init__(self, input_size, output_size, hidden_size) -> None:
        super().__init__()
        self.linear1 = nn.Linear(input_size, hidden_size)
        self.linear2 = nn.Linear(hidden_size, output_size)
    
    def forward(self, x):
        x = F.relu(self.linear1(x))
        x = self.linear2(x)
        return x
    
    def save(self, file_name="model.pth"):
        model_folder_path = "./model"
        if not os.path.exists(model_folder_path):
            os.makedirs(model_folder_path)
        
        file_name = os.path.join(model_folder_path, file_name)
        torch.save(self.state_dict(), file_name)

target_nn = DQN_base_ff(INPUT_SIZE, OUTPUT_SIZE, HIDDEN_SIZE).to(device="mps")

In [44]:
action_test = torch.tensor([[0,0,1,1,0]]).to("mps")
active_pred = target_nn(initial_state.to("mps"))
loss = loss_function(active_prediction=active_pred, gamma=1, immediate_reward=reward, action=action_test, done=done, final_state=final_state, target_nn=target_nn)